# Partitions - Grid 4 (bulk parameters, partitions, spreading)

One notebook for the full per-grid product: full-spectrum bulk parameters (hs, tp, tm02, dp, dm), PTM1 partitions (phs/ptp/dp 0-3), and directional spreading (spr0-3 via `spec_data.dspr()`).

Default range is 1980-2023. Set `years_to_process = [2021]` in the first code cell to run a subset. Existing files are skipped, so a re-run only fills missing variables (including spr if partitions already exist).


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from wavespectra import read_ww3, read_wwm
import os
import glob

# ============================================================================
# Configuration: Paths and Year Range
# ============================================================================
# Input paths
spectra_input_dir = '/lustre/geocean/DATA/GEOOCEAN/BinWavesDuke/grid4'
# spectra_input_dir = '/home/grupos/geocean/montanoj/ShoreShop2026/grid4/outputs/reconstructed_spectra'
uwnd_file = 'inputs/WHACS/north_carolina_04_uwnd_WHACS.nc'
vwnd_file = 'inputs/WHACS/north_carolina_04_vwnd_WHACS.nc'
gebco_file = 'inputs/gebco_bathymetry.nc'

# Output path
output_dir = 'grid4/outputs/output_variables'
os.makedirs(output_dir, exist_ok=True)
# Create subdirectories for each variable
variable_folders = {
    'hs': os.path.join(output_dir, 'hs'),
    'tp': os.path.join(output_dir, 'tp'),
    'tm02': os.path.join(output_dir, 'tm02'),
    'dp': os.path.join(output_dir, 'dp'),
    'dm': os.path.join(output_dir, 'dm'),
    'phs0': os.path.join(output_dir, 'phs0'),
    'phs1': os.path.join(output_dir, 'phs1'),
    'phs2': os.path.join(output_dir, 'phs2'),
    'phs3': os.path.join(output_dir, 'phs3'),
    'ptp0': os.path.join(output_dir, 'ptp0'),
    'ptp1': os.path.join(output_dir, 'ptp1'),
    'ptp2': os.path.join(output_dir, 'ptp2'),
    'ptp3': os.path.join(output_dir, 'ptp3'),
    'dp0': os.path.join(output_dir, 'dp0'),
    'dp1': os.path.join(output_dir, 'dp1'),
    'dp2': os.path.join(output_dir, 'dp2'),
    'dp3': os.path.join(output_dir, 'dp3'),
    'spr0': os.path.join(output_dir, 'spr0'),
    'spr1': os.path.join(output_dir, 'spr1'),
    'spr2': os.path.join(output_dir, 'spr2'),
    'spr3': os.path.join(output_dir, 'spr3'),
}

# Create all variable folders
for var_name, var_dir in variable_folders.items():
    os.makedirs(var_dir, exist_ok=True)
    print(f"Created folder: {var_dir}")


# Year range to process
start_year = 1980
end_year = 2023
# Optional subset. None = all years in [start_year, end_year]
years_to_process = None  # e.g. [2021] or [2010, 2011, 2012]

# Grid name
grid_name = 'grid4'

print("="*60)
print("Multi-Year Wave Partitioning + Spreading")
print("="*60)
print(f"Spectra input directory: {spectra_input_dir}")
print(f"Output directory: {output_dir}")
print(f"Year range: {start_year} - {end_year}")
print("Outputs: hs, tp, tm02, dp, dm, partitions phs/ptp/dp 0-3, spr0-3")
print("="*60)

# Find available spectra files
spectra_files = sorted(glob.glob(f"{spectra_input_dir}/reconstructed_spectra_grid4_*.nc"))
available_years = []
for f in spectra_files:
    year = int(f.split('_')[-1].replace('.nc', ''))
    if start_year <= year <= end_year:
        available_years.append(year)

print(f"\nFound {len(available_years)} spectra files for years {start_year}-{end_year}")
print(f"Available years: {available_years[:5]}...{available_years[-5:] if len(available_years) > 10 else available_years}")

if years_to_process is None:
    years_to_process = list(available_years)
    if available_years:
        print(f"Processing ALL available years: {min(available_years)} - {max(available_years)}")
    else:
        print("No years found")
else:
    invalid_years = [y for y in years_to_process if y not in available_years]
    if invalid_years:
        raise ValueError(
            f"Requested years not found: {invalid_years}. "
            f"Available: {min(available_years)}-{max(available_years)}"
        )
    print(f"Processing specific years: {years_to_process}")
print(f"Total years to process: {len(years_to_process)}")



In [ ]:
# Load wind components (load once, will filter by year in loop)
print("Loading wind components...")
uwnd = xr.open_dataset(uwnd_file)
vwnd = xr.open_dataset(vwnd_file)

print("U wind dataset:")
print(uwnd)
print("\nV wind dataset:")
print(vwnd)

# Get the wind variable names (they might be named differently)
uwnd_var = [v for v in uwnd.data_vars if 'uwnd' in v.lower() or 'u' in v.lower()][0]
vwnd_var = [v for v in vwnd.data_vars if 'vwnd' in v.lower() or 'v' in v.lower()][0]

print(f"\nU wind variable: {uwnd_var}")
print(f"V wind variable: {vwnd_var}")

# Load GEBCO bathymetry (load once, used for all years)
print("\nLoading GEBCO bathymetry...")
gebco = xr.open_dataset(gebco_file)
print("GEBCO dataset loaded.")

In [ ]:
# Function to find closest wind seapoint for each spectra site
def find_closest_wind_seapoints(spectra, uwnd, vwnd):
    """
    Find the closest wind seapoint for each spectra site.
    
    Parameters:
    -----------
    spectra : xarray.Dataset
        Spectra dataset with coord_x and coord_y coordinates
    uwnd : xarray.Dataset
        U-wind dataset  
    vwnd : xarray.Dataset
        V-wind dataset
    
    Returns:
    --------
    seapoint_indices : numpy.ndarray
        Array of seapoint indices, one for each spectra site
    distances_km : numpy.ndarray
        Array of distances in kilometers, one for each spectra site
    distances_deg : numpy.ndarray
        Array of distances in degrees, one for each spectra site
    """
    # Filter wind data to only use years >= 1980 (coordinates for 1979 are wrong)
    print("Filtering wind data to years >= 1980 for coordinate extraction...")
    uwnd_filtered = uwnd.sel(time=slice('1980-01-01', None))
    vwnd_filtered = vwnd.sel(time=slice('1980-01-01', None))
    print(f"Using wind data from {uwnd_filtered.time.min().values} to {uwnd_filtered.time.max().values}")
    
    # Get spectra coordinates
    spectra_lon = spectra.coord_x.values  # Shape: (n_sites,)
    spectra_lat = spectra.coord_y.values  # Shape: (n_sites,)
    
    # Convert spectra lon to 0-360 range if needed
    spectra_lon = np.where(spectra_lon < 0, spectra_lon + 360, spectra_lon)
    
    # Get wind coordinate names (check both coords and data_vars)
    lat_coord = None
    lon_coord = None
    
    # Check coordinates first
    for c in uwnd_filtered.coords:
        if 'lat' in c.lower():
            lat_coord = c
        if 'lon' in c.lower():
            lon_coord = c
    
    # If not found in coords, check data_vars
    if lat_coord is None:
        for v in uwnd_filtered.data_vars:
            if 'lat' in v.lower():
                lat_coord = v
                break
    
    if lon_coord is None:
        for v in uwnd_filtered.data_vars:
            if 'lon' in v.lower():
                lon_coord = v
                break
    
    if lat_coord is None or lon_coord is None:
        print(f"Available coordinates: {list(uwnd_filtered.coords.keys())}")
        print(f"Available data variables: {list(uwnd_filtered.data_vars.keys())}")
        raise ValueError(f"Could not find lat/lon coordinates. Found lat: {lat_coord}, lon: {lon_coord}")
    
    # Get wind coordinate values from filtered dataset
    if lat_coord in uwnd_filtered.coords:
        uwnd_lat = uwnd_filtered[lat_coord].values
    else:
        uwnd_lat = uwnd_filtered[lat_coord].values
    
    if lon_coord in uwnd_filtered.coords:
        uwnd_lon = uwnd_filtered[lon_coord].values
    else:
        uwnd_lon = uwnd_filtered[lon_coord].values
    
    # Handle 2D coordinates (time, seapoint) - use first time step if coordinates don't vary
    if uwnd_lat.ndim > 1:
        if np.std(uwnd_lat, axis=0).max() < 1e-6:
            uwnd_lat = uwnd_lat[0, :]
            uwnd_lon = uwnd_lon[0, :]
        else:
            # Coordinates vary with time - use first time step
            uwnd_lat = uwnd_lat[0, :]
            uwnd_lon = uwnd_lon[0, :]
    
    # Find closest seapoint for each spectra site
    n_sites = len(spectra_lon)
    seapoint_indices = np.zeros(n_sites, dtype=int)
    distances_deg = np.zeros(n_sites)
    distances_km = np.zeros(n_sites)
    
    for site_idx in range(n_sites):
        site_lon = spectra_lon[site_idx]
        site_lat = spectra_lat[site_idx]
        
        # Calculate distances
        lon_diff = np.abs(uwnd_lon - site_lon)
        lon_diff = np.minimum(lon_diff, 360 - lon_diff)  # Handle wrapping
        lat_diff = np.abs(uwnd_lat - site_lat)
        
        dist_deg = np.sqrt(lat_diff**2 + lon_diff**2)
        closest_idx = np.argmin(dist_deg)
        seapoint_indices[site_idx] = closest_idx
        
        # Calculate distance in kilometers
        min_dist_deg = dist_deg[closest_idx]
        distances_deg[site_idx] = min_dist_deg
        
        # Convert to kilometers
        # 1 degree latitude ≈ 111 km
        # 1 degree longitude ≈ 111 km * cos(latitude)
        avg_lat_rad = np.radians((site_lat + uwnd_lat[closest_idx]) / 2)
        lat_km = min_dist_deg * 111
        lon_km = min_dist_deg * 111 * np.cos(avg_lat_rad)
        dist_km = np.sqrt(lat_km**2 + lon_km**2)
        distances_km[site_idx] = dist_km
    
    # Print distance summary
    print(f"\n{'='*60}")
    print(f"Wind-Sea Distance Summary ({n_sites} sites)")
    print(f"{'='*60}")
    print(f"Minimum distance: {distances_km.min():.2f} km ({distances_deg.min():.4f}°)")
    print(f"Maximum distance: {distances_km.max():.2f} km ({distances_deg.max():.4f}°)")
    print(f"Mean distance:    {distances_km.mean():.2f} km ({distances_deg.mean():.4f}°)")
    print(f"Median distance:  {np.median(distances_km):.2f} km ({np.median(distances_deg):.4f}°)")
    print(f"Std deviation:   {distances_km.std():.2f} km ({distances_deg.std():.4f}°)")
    
    # Distance distribution
    print(f"\nDistance distribution:")
    print(f"  < 5 km:    {np.sum(distances_km < 5):4d} sites ({100*np.sum(distances_km < 5)/n_sites:.1f}%)")
    print(f"  5-10 km:   {np.sum((distances_km >= 5) & (distances_km < 10)):4d} sites ({100*np.sum((distances_km >= 5) & (distances_km < 10))/n_sites:.1f}%)")
    print(f"  10-20 km:  {np.sum((distances_km >= 10) & (distances_km < 20)):4d} sites ({100*np.sum((distances_km >= 10) & (distances_km < 20))/n_sites:.1f}%)")
    print(f"  20-50 km:  {np.sum((distances_km >= 20) & (distances_km < 50)):4d} sites ({100*np.sum((distances_km >= 20) & (distances_km < 50))/n_sites:.1f}%)")
    print(f"  >= 50 km:  {np.sum(distances_km >= 50):4d} sites ({100*np.sum(distances_km >= 50)/n_sites:.1f}%)")
    print(f"{'='*60}\n")
    
    return seapoint_indices, distances_km, distances_deg

# Call the function once to create the mapping (will be used in processing loop)
print("Preparing wind-to-spectra site mapping...")
print("(Mapping will be created when spectra are loaded in processing loop)")
print("Note: Wind data will be filtered by year in the processing loop")

In [ ]:
# Extract depth values from GEBCO bathymetry for all spectra coordinates
# Note: Depth is spatial-only (same for all years), so we'll compute it once
# using the first year's spectra coordinates and reuse for all years
print("Extracting depth values (will be reused for all years since it's spatial-only)...")

# Find the depth/elevation variable (could be 'elevation', 'depth', 'z', etc.)
depth_var = None
for var in gebco.data_vars:
    var_lower = var.lower()
    if 'elevation' in var_lower or 'depth' in var_lower or 'bathymetry' in var_lower or var_lower == 'z':
        depth_var = var
        break

if depth_var is None:
    # Use first data variable as fallback
    depth_var = list(gebco.data_vars.keys())[0]
    print(f"Warning: Could not find depth variable, using: {depth_var}")

print(f"\nUsing depth variable: {depth_var}")

# Get coordinate names from GEBCO
gebco_lon = None
gebco_lat = None
for coord in gebco.coords:
    coord_lower = coord.lower()
    if 'lon' in coord_lower or 'x' in coord_lower:
        gebco_lon = coord
    if 'lat' in coord_lower or 'y' in coord_lower:
        gebco_lat = coord

if gebco_lon is None or gebco_lat is None:
    # Check data_vars
    for var in gebco.data_vars:
        var_lower = var.lower()
        if 'lon' in var_lower or 'x' in var_lower:
            gebco_lon = var
        if 'lat' in var_lower or 'y' in var_lower:
            gebco_lat = var

print(f"GEBCO longitude coordinate: {gebco_lon}")
print(f"GEBCO latitude coordinate: {gebco_lat}")

# Helper function to extract depth for spectra
def extract_depth_for_spectra(spectra, gebco):
    """Extract depth values from GEBCO bathymetry for spectra coordinates"""
    # Find the depth/elevation variable
    depth_var = None
    for var in gebco.data_vars:
        var_lower = var.lower()
        if 'elevation' in var_lower or 'depth' in var_lower or 'bathymetry' in var_lower or var_lower == 'z':
            depth_var = var
            break
    if depth_var is None:
        depth_var = list(gebco.data_vars.keys())[0]
    
    # Get coordinate names from GEBCO
    gebco_lon = None
    gebco_lat = None
    for coord in gebco.coords:
        coord_lower = coord.lower()
        if 'lon' in coord_lower or 'x' in coord_lower:
            gebco_lon = coord
        if 'lat' in coord_lower or 'y' in coord_lower:
            gebco_lat = coord
    
    # Get spectra coordinates
    if 'coord_x' in spectra.coords and 'coord_y' in spectra.coords:
        spectra_lon = spectra.coord_x.values
        spectra_lat = spectra.coord_y.values
    else:
        raise ValueError("Could not find coord_x/coord_y in spectra")
    
    # Extract depth values
    gebco_depth = gebco[depth_var]
    if gebco_depth.min() < 0:
        gebco_depth = np.abs(gebco_depth)
    
    depth_at_sites = gebco_depth.sel(
        {gebco_lon: xr.DataArray(spectra_lon, dims='site'),
         gebco_lat: xr.DataArray(spectra_lat, dims='site')},
        method='nearest'
    )
    
    depth_values = depth_at_sites.values if depth_at_sites.ndim == 1 else depth_at_sites.values.flatten()
    
    # Expand to match spectra dimensions
    if 'time' in spectra.dims:
        depth_expanded = xr.DataArray(
            np.broadcast_to(depth_values[np.newaxis, :], (spectra.time.size, len(depth_values))),
            dims=['time', 'site'],
            coords={'time': spectra.time, 'site': spectra.site}
        )
    else:
        depth_expanded = xr.DataArray(
            depth_values,
            dims=['site'],
            coords={'site': spectra.site}
        )
    
    return depth_expanded

# Extract depth once using the first available year's coordinates
# Load first year's spectra to get coordinates
if len(available_years) > 0:
    first_year = available_years[0]
    first_spectra_file = f"{spectra_input_dir}/reconstructed_spectra_grid4_{first_year}.nc"
    if os.path.exists(first_spectra_file):
        print(f"\nLoading first year ({first_year}) spectra to extract depth coordinates...")
        first_spectra = xr.open_dataset(first_spectra_file).rename({'__xarray_dataarray_variable__': 'efth'})
        
        # Extract depth values (spatial-only, 1D array per site)
        depth_values_1d = extract_depth_for_spectra(first_spectra, gebco)
        
        # Store as 1D array (site dimension only) - we'll expand to time dimension for each year
        if 'time' in depth_values_1d.dims:
            # If it has time dimension, take first time step (all should be the same)
            depth_values_1d = depth_values_1d.isel(time=0).drop('time')
        
        # Store site coordinates for later expansion
        depth_sites = depth_values_1d.site
        
        print(f"   Depth extracted: {len(depth_values_1d)} sites")
        print(f"   Depth range: {float(depth_values_1d.min().values):.2f} - {float(depth_values_1d.max().values):.2f} m")
        print(f"   Depth will be reused for all years")
        
        # Close the first spectra file
        first_spectra.close()
    else:
        print(f"   ⚠ Could not load first year file, depth will be extracted per year")
        depth_values_1d = None
        depth_sites = None
else:
    print("   ⚠ No years available, depth will be extracted per year")
    depth_values_1d = None
    depth_sites = None


In [ ]:
# ============================================================================
# Main Processing Loop: Process each year
# ============================================================================

import dask
from dask import delayed
from dask.distributed import Client, get_client
from dask.diagnostics import ProgressBar
import warnings
import logging
import time
from urllib.parse import urlparse

# Helper function to extract depth for spectra
def extract_depth_for_spectra(spectra, gebco):
    """Extract depth values from GEBCO bathymetry for spectra coordinates"""
    # Find the depth/elevation variable
    depth_var = None
    for var in gebco.data_vars:
        var_lower = var.lower()
        if 'elevation' in var_lower or 'depth' in var_lower or 'bathymetry' in var_lower or var_lower == 'z':
            depth_var = var
            break
    if depth_var is None:
        depth_var = list(gebco.data_vars.keys())[0]
    
    # Get coordinate names from GEBCO
    gebco_lon = None
    gebco_lat = None
    for coord in gebco.coords:
        coord_lower = coord.lower()
        if 'lon' in coord_lower or 'x' in coord_lower:
            gebco_lon = coord
        if 'lat' in coord_lower or 'y' in coord_lower:
            gebco_lat = coord
    
    # Get spectra coordinates
    if 'coord_x' in spectra.coords and 'coord_y' in spectra.coords:
        spectra_lon = spectra.coord_x.values
        spectra_lat = spectra.coord_y.values
    else:
        raise ValueError("Could not find coord_x/coord_y in spectra")
    
    # Extract depth values
    gebco_depth = gebco[depth_var]
    if gebco_depth.min() < 0:
        gebco_depth = np.abs(gebco_depth)
    
    depth_at_sites = gebco_depth.sel(
        {gebco_lon: xr.DataArray(spectra_lon, dims='site'),
         gebco_lat: xr.DataArray(spectra_lat, dims='site')},
        method='nearest'
    )
    
    depth_values = depth_at_sites.values if depth_at_sites.ndim == 1 else depth_at_sites.values.flatten()
    
    # Expand to match spectra dimensions
    if 'time' in spectra.dims:
        depth_expanded = xr.DataArray(
            np.broadcast_to(depth_values[np.newaxis, :], (spectra.time.size, len(depth_values))),
            dims=['time', 'site'],
            coords={'time': spectra.time, 'site': spectra.site}
        )
    else:
        depth_expanded = xr.DataArray(
            depth_values,
            dims=['site'],
            coords={'site': spectra.site}
        )
    
    return depth_expanded

# Helper function to expand wind to match spectra dimensions
def expand_wind_for_spectra(wspd_closest, wdir_closest, spectra):
    """Expand wind speed and direction to match spectra spatial dimensions"""
    if 'dpt' in spectra.data_vars:
        target_shape = (spectra.dpt.sizes['time'], spectra.dpt.sizes['site'])
        wspd_broadcast = np.broadcast_to(wspd_closest.values[:, np.newaxis], target_shape)
        wdir_broadcast = np.broadcast_to(wdir_closest.values[:, np.newaxis], target_shape)
        
        wspd_expanded = xr.DataArray(
            wspd_broadcast,
            dims=['time', 'site'],
            coords={'time': spectra.dpt.time, 'site': spectra.dpt.site}
        )
        wdir_expanded = xr.DataArray(
            wdir_broadcast,
            dims=['time', 'site'],
            coords={'time': spectra.dpt.time, 'site': spectra.dpt.site}
        )
        
        for coord_name in spectra.dpt.coords:
            if coord_name not in ['time', 'site']:
                wspd_expanded = wspd_expanded.assign_coords({coord_name: spectra.dpt.coords[coord_name]})
                wdir_expanded = wdir_expanded.assign_coords({coord_name: spectra.dpt.coords[coord_name]})
        
        return wspd_expanded, wdir_expanded
    else:
        raise ValueError("Depth (dpt) must be added to spectra before expanding wind")

# Process each year
print("\n" + "="*60)
print("Starting multi-year processing loop")
print("="*60)

# Suppress warnings and reduce logging verbosity
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Sending large graph')
# Suppress specific RuntimeWarning from wavespectra partition.py about invalid value in scalar divide
warnings.filterwarnings('ignore', message='invalid value encountered in scalar divide')
logging.getLogger('distributed').setLevel(logging.WARNING)
logging.getLogger('distributed.http.proxy').setLevel(logging.ERROR)

# Function to perform PTM1 partitioning
def perform_ptm1_partitioning(spectra, wspd_expanded, wdir_expanded, client=None):
    """Perform PTM1 partitioning on spectra using batched Dask approach"""
    print(f"   Total sites to process: {len(spectra.site)}")

    # Worker configuration
    n_workers = 5
    batch_size = 25
    max_concurrent_batches = 8
    
    # Use provided client or create new one
    use_existing_client = (client is not None)
    if not use_existing_client:
        try:
            client = get_client()
            use_existing_client = True
            print(f"   Using existing Dask client")
        except ValueError:
            pass
    
    if not use_existing_client:
        try:
            import psutil
            total_memory_gb = psutil.virtual_memory().total / (1024**3)
            memory_per_worker_gb = min((total_memory_gb * 0.25) / n_workers, 15.0)
            memory_limit_str = f"{memory_per_worker_gb:.1f}GiB"
        except ImportError:
            memory_limit_str = "15GiB"
        
        try:
            client = Client(processes=True, n_workers=n_workers, memory_limit=memory_limit_str)
            print(f"   Created new Dask client with {n_workers} workers")
        except Exception as e:
            print(f"   Could not create Dask client: {e}, processing sequentially")
            client = None

    n_sites = len(spectra.site)
    n_batches = (n_sites + batch_size - 1) // batch_size
    print(f"   Processing in {n_batches} batches of ~{batch_size} sites each")
    
    def process_site_batch(site_indices, spectra_batch, wspd_batch, wdir_batch, dpt_batch):
        import wavespectra
        import xarray as xr
        import numpy as np
        import warnings
        
        # Suppress warnings in worker processes (they don't inherit main process filters)
        # Use simplefilter for more aggressive suppression
        warnings.simplefilter('ignore', RuntimeWarning)
        warnings.filterwarnings('ignore', message='invalid value encountered in scalar divide')
        warnings.filterwarnings('ignore', message='invalid value encountered in')
        # Also suppress numpy divide by zero warnings at the numpy level
        np.seterr(divide='ignore', invalid='ignore')
        
        batch_results = []
        part_coords = None
        
        for i, site_idx in enumerate(site_indices):
            spectra_site = spectra_batch.isel(site=i)
            wspd_site = wspd_batch.isel(site=i)
            wdir_site = wdir_batch.isel(site=i)
            dpt_site = dpt_batch.isel(site=i)
            
            # Drop all non-dimension coordinates from all three inputs
            # These scalar coordinates (seapoint, station, site, lat, lon, etc.) cause coordinate mismatch
            # wavespectra compares coordinates between wspd, wdir, and dpt
            for var_name, var in [('wspd', wspd_site), ('wdir', wdir_site), ('dpt', dpt_site)]:
                coords_to_drop = [c for c in var.coords.keys() if c not in var.dims]
                if coords_to_drop:
                    if var_name == 'wspd':
                        wspd_site = var.drop_vars(coords_to_drop)
                    elif var_name == 'wdir':
                        wdir_site = var.drop_vars(coords_to_drop)
                    else:
                        dpt_site = var.drop_vars(coords_to_drop)
            
            # Ensure all three have the exact same time coordinate as spectra_site.efth
            # Reindex to match time coordinates exactly (critical for wavespectra)
            if 'time' in spectra_site.efth.coords:
                target_time = spectra_site.efth.time
                if 'time' in wspd_site.coords:
                    wspd_site = wspd_site.reindex(time=target_time, method='nearest')
                if 'time' in wdir_site.coords:
                    wdir_site = wdir_site.reindex(time=target_time, method='nearest')
                if 'time' in dpt_site.coords:
                    dpt_site = dpt_site.reindex(time=target_time, method='nearest')
            
            try:
                dspart_site = spectra_site.spec.partition.ptm1(
                    wspd_site,
                    wdir_site,
                    dpt_site,
                    swells=3,
                    smooth=False,
                )
                if part_coords is None:
                    part_coords = dspart_site.part
                batch_results.append((site_idx, dspart_site))
            except Exception as e:
                site_id = spectra_batch.site.values[i] if hasattr(spectra_batch.site, 'values') else site_idx
                print(f"Error processing site {site_id}: {e}")
                # Ensure part_coords is set (default to 4 partitions)
                if part_coords is None:
                    part_coords = [0, 1, 2, 3]
                
                # Ensure part_coords is a list (might be xarray coordinate)
                if not isinstance(part_coords, list):
                    part_coords = part_coords.values.tolist() if hasattr(part_coords, 'values') else list(part_coords)
                
                # Create empty array matching the working approach from commented code (line 181)
                # Use xr.zeros_like which automatically preserves all coordinates correctly
                n_parts = len(part_coords)
                empty_single = xr.zeros_like(spectra_site.efth.expand_dims('part', axis=0))
                # Expand to n_parts by concatenating (preserves all coordinates automatically)
                empty_efth = xr.concat([empty_single] * n_parts, dim='part')
                empty_efth = empty_efth.assign_coords(part=part_coords)
                dspart_site = empty_efth
                batch_results.append((site_idx, dspart_site))
        
        return batch_results, part_coords

    all_results = []
    
    try:
        if client is not None:
            # Process batches in small groups, creating delayed tasks on-demand
            with ProgressBar():
                for group_start in range(0, n_batches, max_concurrent_batches):
                    group_end = min(group_start + max_concurrent_batches, n_batches)
                    
                    print(f"\nPreparing batches {group_start+1}-{group_end} of {n_batches}...")
                    
                    # Check system load before starting
                    try:
                        import os
                        load_avg = os.getloadavg()[0]
                        cpu_count = os.cpu_count()
                        if load_avg > cpu_count * 1.5:
                            print(f"  ⚠ Warning: High system load ({load_avg:.1f}/{cpu_count} CPUs). This may slow down processing.")
                    except:
                        pass
                    
                    # Create delayed tasks only for this group (with pre-extracted slices)
                    group_delayed = []
                    for batch_idx in range(group_start, group_end):
                        start_idx = batch_idx * batch_size
                        end_idx = min(start_idx + batch_size, n_sites)
                        site_indices = list(range(start_idx, end_idx))
                        
                        # Extract only the slice needed for this batch (small, not 44GB)
                        site_slice = slice(start_idx, end_idx)
                        
                        # # Load data slices with progress indication
                        # print(f"    Loading batch {batch_idx+1} data...", end=' ', flush=True)
                        # try:
                        #     spectra_batch = spectra.isel(site=site_slice).load()  # Load small slice
                        #     print("spectra", end=' ', flush=True)
                        #     wspd_batch = wspd_expanded.isel(site=site_slice).load()
                        #     print("wind", end=' ', flush=True)
                        #     wdir_batch = wdir_expanded.isel(site=site_slice).load()
                        #     print("wdir", end=' ', flush=True)
                        #     dpt_batch = spectra.dpt.isel(site=site_slice).load()
                        #     print("depth ✓", flush=True)
                        # except Exception as e:
                        #     print(f"\n    ✗ Error loading batch {batch_idx+1}: {e}")
                        #     raise
                        # Create lazy slices (don't load yet - let Dask handle loading in workers)
                        # This is much faster as we're not loading data synchronously before submitting to Dask
                        print(f"    Preparing batch {batch_idx+1}...", end=' ', flush=True)
                        try:
                            # Create slices without loading - Dask will load them in workers when needed
                            spectra_batch = spectra.isel(site=site_slice)  # Lazy slice, no .load()
                            wspd_batch = wspd_expanded.isel(site=site_slice)  # Lazy slice
                            wdir_batch = wdir_expanded.isel(site=site_slice)  # Lazy slice
                            dpt_batch = spectra.dpt.isel(site=site_slice)  # Lazy slice
                            print("✓", flush=True)
                        except Exception as e:
                            print(f"\n    ✗ Error preparing batch {batch_idx+1}: {e}")
                            raise
                        
                        # Create delayed task with small slice (not full dataset)
                        delayed_task = delayed(process_site_batch)(
                            site_indices, spectra_batch, wspd_batch, wdir_batch, dpt_batch
                        )
                        group_delayed.append(delayed_task)
                        
                        # Clear references immediately
                        del spectra_batch, wspd_batch, wdir_batch, dpt_batch
                    
                    print(f"Submitting batches {group_start+1}-{group_end} to Dask...")
                    
                    # Compute this group
                    group_futures = [client.compute(d) for d in group_delayed]
                    
                    # Wait for this group to complete before starting next
                    try:
                        group_results = client.gather(group_futures)
                        all_results.extend(group_results)
                        print(f"  ✓ Completed batches {group_start+1}-{group_end}")
                    except Exception as e:
                        print(f"  ✗ Error in batches {group_start+1}-{group_end}: {e}")
                        # Try to recover what we can from this group
                        for i, future in enumerate(group_futures):
                            try:
                                if future.status == 'finished':
                                    all_results.append(client.gather([future])[0])
                                    print(f"    Recovered batch {group_start + i + 1}")
                            except:
                                pass
                        # Continue with next group instead of failing completely
                        continue
                    
                    # Clear delayed tasks to free memory
                        del group_delayed, group_futures
                    
                    # Small delay between groups to let memory settle
                        time.sleep(1.0)
        else:
            # Fallback: process sequentially if no client
                        print("No Dask client available, processing sequentially...")
                        for batch_idx in range(n_batches):
                            start_idx = batch_idx * batch_size
                            end_idx = min(start_idx + batch_size, n_sites)
                            site_indices = list(range(start_idx, end_idx))
                
                            print(f"Processing batch {batch_idx+1}/{n_batches}...")
                
                            # Extract slice and process directly
                            site_slice = slice(start_idx, end_idx)
                            spectra_batch = spectra.isel(site=site_slice).load()
                            wspd_batch = wspd_expanded.isel(site=site_slice).load()
                            wdir_batch = wdir_expanded.isel(site=site_slice).load()
                            dpt_batch = spectra.dpt.isel(site=site_slice).load()
                            
                            result = process_site_batch(site_indices, spectra_batch, wspd_batch, wdir_batch, dpt_batch)
                            all_results.append(result)
                            
                            del spectra_batch, wspd_batch, wdir_batch, dpt_batch
                            
    except KeyboardInterrupt:
        print("\nComputation interrupted by user")
        if all_results:
            print(f"Got {len(all_results)}/{n_batches} batches before interruption")
        raise
    except Exception as e:
        print(f"\nError during computation: {e}")
        if all_results:
            print(f"Got {len(all_results)}/{n_batches} batches before error")
        print("This may be due to memory issues. Try:")
        print(f"  1. Reduce batch_size (currently {batch_size})")
        print(f"  2. Reduce n_workers (currently {n_workers})")
        print(f"  3. Reduce max_concurrent_batches (currently {max_concurrent_batches})")
        print("  4. Close other applications to free memory")
        if len(all_results) == 0:
            raise
    
    # Combine partition results (outside try/except, but still in function)
    if len(all_results) == 0:
        raise RuntimeError("No results obtained. Computation failed or was cancelled.")

    if len(all_results) < n_batches:
        print(f"   WARNING: Only got {len(all_results)}/{n_batches} batches, proceeding with partial results...")

    # Combine partition results
    all_batch_results = []
    part_coords = None
    for batch_results, batch_part_coords in all_results:
        all_batch_results.extend(batch_results)
        if part_coords is None and batch_part_coords is not None:
            part_coords = batch_part_coords

    all_batch_results.sort(key=lambda x: x[0])

    efth_list = []
    for site_idx, ds in all_batch_results:
        site_id = spectra.site.values[site_idx]
        if isinstance(ds, xr.Dataset):
            efth = ds.efth
        else:
            efth = ds
        efth_with_site = efth.expand_dims('site').assign_coords(site=[site_id])
        efth_list.append(efth_with_site)

    dspart_efth = xr.concat(efth_list, dim='site')
    dspart = xr.Dataset({'efth': dspart_efth})
    dspart = dspart.assign_coords(site=spectra.site)
    
    if part_coords is not None:
        if hasattr(part_coords, 'values'):
            part_values = part_coords.values
        elif hasattr(part_coords, '__iter__') and not isinstance(part_coords, str):
            part_values = list(part_coords)
        else:
            part_values = part_coords
        
        if 'part' not in dspart.coords:
            dspart = dspart.assign_coords(part=('part', part_values))

    return dspart

# Function to check if all partitioned files already exist for a year
def check_partitions_exist(year, grid_name, variable_folders):
    """Check if all partitioned files (phs0-3, ptp0-3, dp0-3, spr0-3) already exist for a given year"""
    partition_names = ['phs0', 'phs1', 'phs2', 'phs3']
    all_exist = True
    
    for i, part_name in enumerate(partition_names):
        hs_file = os.path.join(variable_folders[part_name], f'nc_{part_name}_{grid_name}_{year}.nc')
        tp_file = os.path.join(variable_folders[f'ptp{i}'], f'nc_ptp{i}_{grid_name}_{year}.nc')
        dp_file = os.path.join(variable_folders[f'dp{i}'], f'nc_dp{i}_{grid_name}_{year}.nc')
        spr_file = os.path.join(variable_folders[f'spr{i}'], f'nc_spr{i}_{grid_name}_{year}.nc')
        
        if not (os.path.exists(hs_file) and os.path.exists(tp_file) and os.path.exists(dp_file) and os.path.exists(spr_file)):
            all_exist = False
            break
    
    return all_exist

# Function to check if a year is completely processed (all files exist)
def check_year_complete(year, grid_name, variable_folders):
    """Check if all files for a year already exist (partitioned + full spectrum)"""
    # Check partitioned files
    partitions_exist = check_partitions_exist(year, grid_name, variable_folders)
    
    # Check full spectrum files
    full_spectrum_params = ['hs', 'tp', 'tm02', 'dp', 'dm']
    all_full_exist = True
    for param_name in full_spectrum_params:
        output_file = os.path.join(variable_folders[param_name], f'nc_{param_name}_{grid_name}_{year}.nc')
        if not os.path.exists(output_file):
            all_full_exist = False
            break
    
    return partitions_exist and all_full_exist

# Function to save partitioned data
def save_partitioned_data(dspart, spectra, year, grid_name, output_dir, variable_folders):
    """Save partitioned data (hs, tp, dp, spr) for each partition - only partitions 0-3
    FIXED: Converts all data to pure numpy arrays before computing bulk parameters to avoid Dask memory issues
    Saves files to appropriate variable folders and checks for existing files before computing
    If dspart is None, only checks for existing files and skips computation"""
    partition_names = ['phs0', 'phs1', 'phs2', 'phs3']
    
    for i, part_name in enumerate(partition_names):
        # Define output files in their respective folders
        hs_file = os.path.join(variable_folders[part_name], f'nc_{part_name}_{grid_name}_{year}.nc')
        tp_file = os.path.join(variable_folders[f'ptp{i}'], f'nc_ptp{i}_{grid_name}_{year}.nc')
        dp_file = os.path.join(variable_folders[f'dp{i}'], f'nc_dp{i}_{grid_name}_{year}.nc')
        spr_file = os.path.join(variable_folders[f'spr{i}'], f'nc_spr{i}_{grid_name}_{year}.nc')
        
        # Check if all files already exist - skip entire partition if so
        if os.path.exists(hs_file) and os.path.exists(tp_file) and os.path.exists(dp_file) and os.path.exists(spr_file):
            print(f"    Partition {i} ({part_name}): All files already exist, skipping... ✓")
            continue
        
        # If dspart is None, we can't compute, so skip
        if dspart is None:
            print(f"    Partition {i} ({part_name}): Missing files but no partitioned data available, skipping...")
            print(f"      (Run partitioning step to generate missing files)")
            continue
        
        if i >= len(dspart.part):
            print(f"    Partition {i} ({part_name}): Not available in partitioned data, skipping...")
            continue
            
        print(f"    Processing partition {i} ({part_name})...")
        
        try:
            # Extract partition data - dspart should already be fully loaded
            part_data = dspart.isel(part=i)
            # CRITICAL: Convert efth to pure numpy array before creating spec object
            # This prevents wavespectra from creating new Dask arrays
            if hasattr(part_data.efth, 'chunks') and part_data.efth.chunks is not None:
                part_data = part_data.load()
                
                # Convert efth to pure numpy array (no Dask chunks)
                efth_numpy = part_data.efth.values if hasattr(part_data.efth, 'values') else np.array(part_data.efth)
                
                # Create new DataArray from numpy array (no Dask backend)
                efth_clean = xr.DataArray(
                    efth_numpy,
                    dims=part_data.efth.dims,
                    coords=part_data.efth.coords,
                    attrs=part_data.efth.attrs
                )
                
                part_ds = xr.Dataset({'efth': efth_clean})
            else:
                # Data already loaded, create dataset directly
                part_ds = xr.Dataset({'efth': part_data.efth})
            
            spec_data = part_ds.spec
        except Exception as e:
            print(f"      ✗ Error preparing partition data: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        # Compute and save hs (only if file doesn't exist)
        if not os.path.exists(hs_file):
            try:
                print(f"      Computing hs...", end=' ', flush=True)
                hs = spec_data.hs()
                # Ensure it's numpy (should already be, but double-check)
                if hasattr(hs, 'chunks') and hs.chunks is not None:
                    hs = hs.load()  # Use load() instead of compute() to avoid Dask client
                hs_values = hs.values if hasattr(hs, 'values') else np.array(hs)
                hs = xr.DataArray(hs_values, dims=hs.dims, coords=hs.coords)
                print("✓", flush=True)
                
                # Save hs
                print(f"      Saving hs...", end=' ', flush=True)
                hs_ds = xr.Dataset({'hs': hs})
                hs_ds = hs_ds.assign_coords(site=part_data.site, time=part_data.time)
                if hasattr(spectra, 'attrs'):
                    hs_ds.attrs.update(spectra.attrs)
                hs_ds.attrs.update({
                    'partition': part_name,
                    'partition_method': 'PTM1',
                    'variable': 'significant_wave_height',
                    'units': 'm'
                })
                # Data is already loaded, so save directly with maximum compression
                hs_ds.to_netcdf(hs_file, encoding={'hs': {'zlib': True, 'complevel': 6, 'shuffle': True}})
                print(f"✓ Saved to: {os.path.basename(hs_file)}")
                
                # Clear hs from memory
                del hs, hs_ds
            except Exception as e:
                print(f"✗ Error computing/saving hs: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"      hs already exists, skipping... ✓")
        
        # Compute and save tp (only if file doesn't exist)
        if not os.path.exists(tp_file):
            try:
                print(f"      Computing tp...", end=' ', flush=True)
                tp = spec_data.tp()
                if hasattr(tp, 'chunks') and tp.chunks is not None:
                    tp = tp.load()  # Use load() instead of compute()
                tp_values = tp.values if hasattr(tp, 'values') else np.array(tp)
                tp = xr.DataArray(tp_values, dims=tp.dims, coords=tp.coords)
                print("✓", flush=True)
                
                # Save tp
                print(f"      Saving tp...", end=' ', flush=True)
                tp_ds = xr.Dataset({'tp': tp})
                tp_ds = tp_ds.assign_coords(site=part_data.site, time=part_data.time)
                if hasattr(spectra, 'attrs'):
                    tp_ds.attrs.update(spectra.attrs)
                tp_ds.attrs.update({
                    'partition': part_name,
                    'partition_method': 'PTM1',
                    'variable': 'peak_period',
                    'units': 's'
                })
                tp_ds.to_netcdf(tp_file, encoding={'tp': {'zlib': True, 'complevel': 6, 'shuffle': True}})
                print(f"✓ Saved to: {os.path.basename(tp_file)}")
                
                # Clear tp from memory
                del tp, tp_ds
            except Exception as e:
                print(f"✗ Error computing/saving tp: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"      tp already exists, skipping... ✓")
        
        # Compute and save dp (only if file doesn't exist)
        if not os.path.exists(dp_file):
            try:
                print(f"      Computing dp...", end=' ', flush=True)
                dp = spec_data.dpm()
                if hasattr(dp, 'chunks') and dp.chunks is not None:
                    dp = dp.load()  # Use load() instead of compute()
                dp_values = dp.values if hasattr(dp, 'values') else np.array(dp)
                dp = xr.DataArray(dp_values, dims=dp.dims, coords=dp.coords)
                print("✓", flush=True)
                
                # Save dp
                print(f"      Saving dp...", end=' ', flush=True)
                dp_ds = xr.Dataset({'dp': dp})
                dp_ds = dp_ds.assign_coords(site=part_data.site, time=part_data.time)
                if hasattr(spectra, 'attrs'):
                    dp_ds.attrs.update(spectra.attrs)
                dp_ds.attrs.update({
                    'partition': part_name,
                    'partition_method': 'PTM1',
                    'variable': 'peak_direction',
                    'units': 'degrees'
                })
                dp_ds.to_netcdf(dp_file, encoding={'dp': {'zlib': True, 'complevel': 6, 'shuffle': True}})
                print(f"✓ Saved to: {os.path.basename(dp_file)}")
                
                # Clear dp and partition data from memory
                del dp, dp_ds
            except Exception as e:
                print(f"✗ Error computing/saving dp: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"      dp already exists, skipping... ✓")

        # Compute and save spr (spreading) - only if file doesn't exist
        if not os.path.exists(spr_file):
            try:
                print(f"      Computing spr (spreading)...", end=' ', flush=True)
                spr = spec_data.dspr()
                if hasattr(spr, 'chunks') and spr.chunks is not None:
                    spr = spr.load()
                spr_values = spr.values if hasattr(spr, 'values') else np.array(spr)
                spr = xr.DataArray(spr_values, dims=spr.dims, coords=spr.coords)
                print("✓", flush=True)

                print(f"      Saving spr...", end=' ', flush=True)
                spr_ds = xr.Dataset({'spr': spr})
                spr_ds = spr_ds.assign_coords(site=part_data.site, time=part_data.time)
                if hasattr(spectra, 'attrs'):
                    spr_ds.attrs.update(spectra.attrs)
                spr_ds.attrs.update({
                    'partition': part_name,
                    'partition_method': 'PTM1',
                    'variable': 'directional_spreading',
                    'units': 'degrees'
                })
                spr_ds.to_netcdf(spr_file, encoding={'spr': {'zlib': True, 'complevel': 6, 'shuffle': True}})
                print(f"✓ Saved to: {os.path.basename(spr_file)}")

                del spr, spr_ds
            except Exception as e:
                print(f"✗ Error computing/saving spr: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"      spr already exists, skipping... ✓")

        # Clean up partition data
        del part_data, part_ds, spec_data
        
        print(f"     ✓ Completed partition {i} ({part_name})")
        
        # Force garbage collection to free memory
        import gc
        gc.collect()

# Function to save bulk parameters for full spectrum
def save_full_spectrum_bulk_params(spectra, year, grid_name, output_dir, variable_folders, spectra_file_path=None):
    """Save bulk parameters (hs, tp, tm02, dp, dm) for the entire original spectrum
    Each parameter is computed and saved immediately to prevent memory buildup and allow resuming.
    The spectra is reloaded from disk for each parameter to ensure clean memory state.
    Saves files to appropriate variable folders and checks for existing files before computing.
    
    Args:
        spectra: xarray Dataset (used only for coordinates/attrs, not loaded)
        year: year being processed
        grid_name: grid name
        output_dir: output directory
        variable_folders: dictionary mapping variable names to their output folders
        spectra_file_path: Optional path to spectra file. If provided, reloads from disk.
                          If None, uses spectra object (less memory efficient)
    """
    import gc
    import os
    import xarray as xr
    
    # First, check which parameters already exist
    params = [
        ('hs', 'hs', 'significant_wave_height', 'm', 'Significant wave height computed from full spectrum (non-partitioned)'),
        ('tp', 'tp', 'peak_period', 's', 'Peak period computed from full spectrum (non-partitioned)'),
        ('tm02', 'tm02', 'mean_period', 's', 'Mean wave period (Tm02) computed from full spectrum (non-partitioned)'),
        ('dp', 'dpm', 'peak_direction', 'degrees', 'Peak direction computed from full spectrum (non-partitioned)'),
        ('dm', 'dm', 'mean_direction', 'degrees', 'Mean wave direction computed from full spectrum (non-partitioned)')
    ]
    
    # Check which files exist
    existing_params = []
    missing_params = []
    for param_name, _, _, _, _ in params:
        output_file = os.path.join(variable_folders[param_name], f'nc_{param_name}_{grid_name}_{year}.nc')
        if os.path.exists(output_file):
            existing_params.append(param_name)
        else:
            missing_params.append(param_name)
    
    # Print status
    if existing_params:
        print(f"    Full spectrum parameters already exist: {', '.join(existing_params)} ✓")
    if missing_params:
        print(f"    Computing missing full spectrum parameters: {', '.join(missing_params)}...")
    else:
        print(f"    All full spectrum parameters already exist, skipping computation ✓")
        return  # Early return if all files exist
    
    # Force cleanup before starting
    gc.collect()
    
    # Get spectra file path if not provided (try to infer from spectra object)
    if spectra_file_path is None:
        # Try to get file path from spectra if it's still open
        if hasattr(spectra, 'encoding') and 'source' in spectra.encoding:
            spectra_file_path = spectra.encoding['source']
        elif hasattr(spectra, 'efth') and hasattr(spectra.efth, 'encoding') and 'source' in spectra.efth.encoding:
            spectra_file_path = spectra.efth.encoding['source']
    
    # Store coordinates and attrs before closing spectra (if we have file path)
    if spectra_file_path and os.path.exists(spectra_file_path):
        # Only close if we have a valid file path
        if spectra is not None:
            site_coords = spectra.site
            time_coords = spectra.time
            spectra_attrs = spectra.attrs if hasattr(spectra, 'attrs') else {}
            # Close the spectra dataset to free memory
            if hasattr(spectra, 'close'):
                spectra.close()
            del spectra
            gc.collect()
            print("    Closed spectra dataset, will reload from disk for each parameter")
    else:
        # If no file path, we need to keep spectra object
        if spectra is not None:
            site_coords = spectra.site
            time_coords = spectra.time
            spectra_attrs = spectra.attrs if hasattr(spectra, 'attrs') else {}
    
    # Process each parameter: load spectra from disk -> compute -> save -> delete -> cleanup
    # Only process missing parameters
    for param_name, method_name, var_name, units, description in params:
        # Save to the appropriate variable folder
        output_file = os.path.join(variable_folders[param_name], f'nc_{param_name}_{grid_name}_{year}.nc')
        
        # Check if file already exists (resume capability)
        if os.path.exists(output_file):
            print(f"      {param_name} already exists, skipping... ✓")
            continue
        
        # Load spectra fresh from disk for this parameter (critical for memory management)
        spectra_loaded = None
        spectra_ds = None
        spec_data = None
        
        try:
            # Process in chunks by site to reduce memory usage (especially important for dp/dm)
            # This is critical for large datasets with many sites
            n_sites_total = len(site_coords) if spectra_file_path else len(spectra.site)
            site_chunk_size = 200  # Process 200 sites at a time (adjust based on memory)
            
            # For directional parameters (dp, dm), use smaller chunks
            if param_name in ['dp', 'dm']:
                site_chunk_size = 100  # Smaller chunks for memory-intensive directional params
            
            n_chunks = (n_sites_total + site_chunk_size - 1) // site_chunk_size
            
            if n_chunks > 1:
                print(f"      Computing {param_name} in {n_chunks} chunks of ~{site_chunk_size} sites each...", end=' ', flush=True)
            else:
                print(f"      Computing {param_name}...", end=' ', flush=True)
            
            param_chunks = []
            
            for chunk_idx in range(n_chunks):
                start_site = chunk_idx * site_chunk_size
                end_site = min(start_site + site_chunk_size, n_sites_total)
                
                # Reload spectra chunk from disk (most memory efficient)
                if spectra_file_path and os.path.exists(spectra_file_path):
                    # Open with chunks, then select site range
                    spectra_chunk = xr.open_dataset(spectra_file_path, chunks={'time': 1000}).rename({'__xarray_dataarray_variable__': 'efth'})
                    spectra_chunk = spectra_chunk.isel(site=slice(start_site, end_site))
                    # Load only this chunk into memory
                    spectra_chunk = spectra_chunk.load()
                else:
                    # Fallback: use provided spectra object
                    if spectra is not None:
                        spectra_chunk = spectra.isel(site=slice(start_site, end_site))
                        spectra_chunk = spectra_chunk.load() if hasattr(spectra_chunk.efth, 'chunks') else spectra_chunk
                    else:
                        raise ValueError("Cannot process: no spectra file path and spectra object is None")
                
                spectra_ds_chunk = xr.Dataset({'efth': spectra_chunk.efth})
                spec_data_chunk = spectra_ds_chunk.spec
                
                # Compute parameter for this chunk
                param_method = getattr(spec_data_chunk, method_name)
                param_chunk = param_method().load()
                param_chunks.append(param_chunk)
                
                # Clean up chunk data immediately
                del spectra_chunk, spectra_ds_chunk, spec_data_chunk
                gc.collect()
                
                if n_chunks > 1:
                    print(f"{chunk_idx+1}/{n_chunks}...", end=' ', flush=True)
            
            # Concatenate all chunks along site dimension
            if len(param_chunks) > 1:
                # Ensure all chunks have proper site coordinates before concatenating
                for i, chunk in enumerate(param_chunks):
                    if 'site' not in chunk.coords or len(chunk.coords['site']) != len(chunk.site):
                        # Reassign site coordinates from original
                        start_site = i * site_chunk_size
                        end_site = min(start_site + site_chunk_size, n_sites_total)
                        if spectra_file_path:
                            chunk = chunk.assign_coords(site=site_coords[start_site:end_site])
                        else:
                            chunk = chunk.assign_coords(site=spectra.site[start_site:end_site])
                        param_chunks[i] = chunk
                
                param_value = xr.concat(param_chunks, dim='site')
                del param_chunks
                gc.collect()
            else:
                param_value = param_chunks[0]
                del param_chunks
            
            # Ensure param_value has correct site coordinates
            if 'site' not in param_value.coords or len(param_value.coords['site']) != n_sites_total:
                param_value = param_value.assign_coords(site=site_coords if spectra_file_path else spectra.site)
            
            print("✓", flush=True)
            
            # CRITICAL: Save immediately after computation, before any other operations
            # This ensures we don't lose the computation if memory issues occur
            print(f"      Saving {param_name}...", end=' ', flush=True)
            
            # Create dataset and save in a separate try block to catch save errors
            try:
                param_ds = xr.Dataset({param_name: param_value})
                
                # Use stored coordinates if we reloaded from disk
                if spectra_file_path:
                    param_ds = param_ds.assign_coords(site=site_coords, time=time_coords)
                    param_ds.attrs.update(spectra_attrs)
                else:
                    param_ds = param_ds.assign_coords(site=spectra.site, time=spectra.time)
                    if hasattr(spectra, 'attrs'):
                        param_ds.attrs.update(spectra.attrs)
                
                param_ds.attrs.update({
                    'variable': var_name,
                    'units': units,
                    'description': description
                })
                
                # Save to disk with explicit flush
                param_ds.to_netcdf(
                    output_file, 
                    encoding={param_name: {'zlib': True, 'complevel': 6, 'shuffle': True}},
                    mode='w'  # Explicitly overwrite mode
                )
                
                # Verify file was written successfully
                if not os.path.exists(output_file):
                    raise IOError(f"File {output_file} was not created after save")
                
                # Check file size to ensure it's not empty
                file_size = os.path.getsize(output_file)
                if file_size == 0:
                    raise IOError(f"File {output_file} is empty after save")
                
                print(f"✓ Saved to: {os.path.basename(output_file)} ({file_size / 1024**2:.1f} MB)")
                
                # Only delete from memory AFTER successful save
                del param_ds
                
            except Exception as save_error:
                print(f"\n      ✗ Error saving {param_name}: {save_error}")
                # Keep param_value in memory so we can retry saving
                # Don't delete it yet - user might want to retry
                raise  # Re-raise to stop and allow retry
            
            # Now safe to delete computed value (it's saved to disk)
            del param_value
            
            # Clean up - no need to close spectra_loaded since we processed in chunks
            # Each chunk was already cleaned up during processing
            # Force garbage collection after each parameter
            gc.collect()
            
        except Exception as e:
            print(f"\n      ✗ Error computing/saving {param_name}: {e}")
            # Clean up on error
            if 'param_value' in locals():
                del param_value
            if 'param_ds' in locals():
                del param_ds
            if 'param_chunks' in locals():
                del param_chunks
            if 'spectra_chunk' in locals():
                if hasattr(spectra_chunk, 'close'):
                    spectra_chunk.close()
                del spectra_chunk
            gc.collect()
            raise  # Re-raise to stop processing if critical
    
    print("    ✓ Completed saving full spectrum bulk parameters")

# ============================================================================
# Process each year in the loop
# ============================================================================

# ============================================================================
# Pre-compute wind-to-spectra mapping (once, before processing all years)
# ============================================================================
# Since spectra coordinates don't change between years, we can compute the
# mapping once using the first year's spectra and reuse it for all years
print("\n" + "="*60)
print("Pre-computing wind-to-spectra site mapping")
print("="*60)

if len(available_years) > 0:
    first_year = available_years[0]
    first_spectra_file = f"{spectra_input_dir}/reconstructed_spectra_grid4_{first_year}.nc"
    
    if os.path.exists(first_spectra_file):
        print(f"Loading first year ({first_year}) spectra to compute mapping...")
        first_spectra = xr.open_dataset(first_spectra_file).rename({'__xarray_dataarray_variable__': 'efth'})
        
        # Compute mapping using full wind dataset (all years)
        # The mapping only depends on coordinates, not on time
        print("Computing wind-to-spectra mapping...")
        seapoint_indices, distances_km, distances_deg = find_closest_wind_seapoints(first_spectra, uwnd, vwnd)
        
        print(f"\n✓ Mapping computed! Will be reused for all {len(available_years)} years")
        print(f"   Mean distance: {distances_km.mean():.2f} km")
        print(f"   Max distance: {distances_km.max():.2f} km")
        
        # Close the first spectra file
        first_spectra.close()
    else:
        print(f"⚠ Could not load first year file, mapping will be computed per year")
        seapoint_indices = None
else:
    print("⚠ No years available")
    seapoint_indices = None

print("="*60 + "\n")

# ============================================================================
# Main Processing Loop: Process each year
# ============================================================================
for year_idx, year in enumerate(years_to_process):
    print("\n" + "="*80)
    print(f"PROCESSING YEAR {year} ({year_idx+1}/{len(years_to_process)})")
    print("="*80)
    
    try:
        # Check if year is already completely processed - skip if so
        year_complete = check_year_complete(year, grid_name, variable_folders)
        if year_complete:
            print(f"\n✓ Year {year} is already completely processed (all files exist), skipping...")
            continue
        else:
            # Debug: show which files are missing
            print(f"\n⚠ Year {year} is not complete, will process missing files...")
            partitions_exist = check_partitions_exist(year, grid_name, variable_folders)
            if not partitions_exist:
                print(f"   Missing: Partitioned files (phs0-3, ptp0-3, dp0-3, spr0-3)")
            full_spectrum_params = ['hs', 'tp', 'tm02', 'dp', 'dm']
            missing_full = []
            for param_name in full_spectrum_params:
                output_file = os.path.join(variable_folders[param_name], f'nc_{param_name}_{grid_name}_{year}.nc')
                if not os.path.exists(output_file):
                    missing_full.append(param_name)
            if missing_full:
                print(f"   Missing: Full spectrum files ({', '.join(missing_full)})")
        
        # Load spectra for this year
        spectra_file = f"{spectra_input_dir}/reconstructed_spectra_grid4_{year}.nc"
        if not os.path.exists(spectra_file):
            print(f"  ⚠ Spectra file not found: {spectra_file}")
            continue
        
        print(f"\n1. Loading spectra for {year}...")
        spectra = xr.open_dataset(spectra_file).rename({'__xarray_dataarray_variable__': 'efth'})
        print(f"   Spectra shape: {spectra.efth.shape}")
        print(f"   Time range: {spectra.time.min().values} to {spectra.time.max().values}")
        
        # Add depth to spectra (reuse pre-extracted depth, expand to match time dimension)
        print(f"\n2. Adding depth values (reusing pre-extracted spatial depth)...")
        if depth_values_1d is not None:
            # Expand depth to match this year's time dimension
            depth_expanded = xr.DataArray(
                np.broadcast_to(depth_values_1d.values[np.newaxis, :], (spectra.time.size, len(depth_values_1d))),
                dims=['time', 'site'],
                coords={'time': spectra.time, 'site': spectra.site}
            )
            spectra['dpt'] = depth_expanded
            print(f"   Depth shape: {spectra.dpt.shape} (expanded from {len(depth_values_1d)} sites)")
        else:
            # Fallback: extract depth if not pre-extracted
            print(f"   Extracting depth for this year (fallback)...")
            spectra['dpt'] = extract_depth_for_spectra(spectra, gebco)
            print(f"   Depth shape: {spectra.dpt.shape}")
        
        # Filter wind data to current year
        print(f"\n3. Filtering wind data to {year}...")
        uwnd_year = uwnd.sel(time=slice(f'{year}-01-01', f'{year}-12-31'))
        vwnd_year = vwnd.sel(time=slice(f'{year}-01-01', f'{year}-12-31'))
        print(f"   Wind data filtered: {len(uwnd_year.time)} time steps")
        
        # Use pre-computed mapping or compute if needed
        if seapoint_indices is None:
            print(f"\n4. Computing wind-to-spectra mapping for {year}...")
            seapoint_indices, distances_km, distances_deg = find_closest_wind_seapoints(spectra, uwnd_year, vwnd_year)
            print(f"   Mapped {len(seapoint_indices)} sites to wind seapoints")
        else:
            print(f"\n4. Using pre-computed wind-to-spectra mapping...")
            print(f"   Using mapping for {len(seapoint_indices)} sites")
        
        # Extract wind data for each site (using its closest seapoint)
        n_sites = len(spectra.site)
        n_times = len(spectra.time)
        
        wspd_sites = np.zeros((n_times, n_sites))
        wdir_sites = np.zeros((n_times, n_sites))
        
        for site_idx in range(n_sites):
            seapoint_idx = seapoint_indices[site_idx]
            u_site = uwnd_year[uwnd_var].isel(seapoint=seapoint_idx)
            v_site = vwnd_year[vwnd_var].isel(seapoint=seapoint_idx)
            
            # Compute wind speed and direction
            wspd_site = np.sqrt(u_site**2 + v_site**2)
            wdir_site = (np.arctan2(u_site, v_site) * 180 / np.pi + 180) % 360
            
            # Align wind time with spectra time
            wspd_aligned = wspd_site.reindex(time=spectra.time, method='nearest')
            wdir_aligned = wdir_site.reindex(time=spectra.time, method='nearest')
            
            wspd_sites[:, site_idx] = wspd_aligned.values
            wdir_sites[:, site_idx] = wdir_aligned.values
        
        # Create DataArrays with site-specific wind data
        wspd_expanded = xr.DataArray(
            wspd_sites,
            dims=['time', 'site'],
            coords={'time': spectra.time, 'site': spectra.site}
        )
        wdir_expanded = xr.DataArray(
            wdir_sites,
            dims=['time', 'site'],
            coords={'time': spectra.time, 'site': spectra.site}
        )
        
        print(f"   Wind speed shape: {wspd_expanded.shape}")
        print(f"   Wind direction shape: {wdir_expanded.shape}")
        print(f"   Wind speed shape: {wspd_expanded.shape}")
        print(f"   Wind direction shape: {wdir_expanded.shape}")
        
        # Check if partitioned files already exist
        partitions_exist = check_partitions_exist(year, grid_name, variable_folders)
        
        if partitions_exist:
            print(f"\n5. Partitioned files already exist for {year}, skipping PTM1 partitioning (no Dask needed)...")
            print(f"   ✓ All partitioned files found, will only save missing variables if any")
            dspart = None  # Not needed since files exist
        else:
            # Perform PTM1 partitioning (uses Dask)
            print(f"\n5. Performing PTM1 partitioning for {year}...")
            dspart = perform_ptm1_partitioning(spectra, wspd_expanded, wdir_expanded, client=None)
            print(f"   Partitioning complete! Shape: {dspart.efth.shape}")
            
            # CRITICAL FIX: Ensure dspart is fully computed (not Dask) before saving
            print(f"   Ensuring partitioned data is fully computed...", end=' ', flush=True)
            if hasattr(dspart.efth, 'chunks') and dspart.efth.chunks is not None:
                dspart = dspart.load()  # Force computation of all Dask arrays
            print("✓", flush=True)
            
            # CRITICAL FIX: Close Dask client BEFORE saving to prevent memory issues
            # The save functions now work with pure numpy arrays, so Dask client is not needed
            try:
                from dask.distributed import get_client
                client = get_client()
                print(f"   Closing Dask client before saving...", end=' ', flush=True)
                client.close()
                print("✓", flush=True)
            except ValueError:
                pass  # No client exists
        
        # Save partitioned data (partitions 0-3 for hs, tp, dp, spr)
        # This function will check for existing files and skip computation if they exist
        # All data is now pure numpy arrays, no Dask operations will occur
        print(f"\n6. Saving partitioned data for {year}...")
        if dspart is not None:
            # Only call save function if we have partitioned data (files don't exist)
            save_partitioned_data(dspart, spectra, year, grid_name, output_dir, variable_folders)
        else:
            # Files already exist, just check and save any missing ones
            save_partitioned_data(None, spectra, year, grid_name, output_dir, variable_folders)
        
        # Save bulk parameters for full spectrum (hs, tp, tm02, dp, dm)
        # This function checks which files exist and only computes missing ones
        print(f"\n7. Saving bulk parameters for full spectrum ({year})...")
        # Pass spectra file path so function can reload from disk (more memory efficient)
        # Note: spectra object might be closed if partitions were computed, but file path is always available
        save_full_spectrum_bulk_params(spectra, year, grid_name, output_dir, variable_folders, spectra_file_path=spectra_file)
        
        print(f"\n   ✓ Year {year} processing completed successfully!")
        
        # Clean up memory
        del spectra, wspd_expanded, wdir_expanded
        if dspart is not None:
            del dspart
        import gc
        gc.collect()
        
    except Exception as e:
        print(f"\n  ✗ Error processing year {year}: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*80)
print("Multi-year processing completed (bulk params + partitions + spreading)!")
print("="*80)



In [ ]:
buoys = {
     '41108': (-78.016, 33.721), 
     'ssbn7': (-78.484, 33.838),
     '41119': (-78.483, 33.842),
     '41013': (-77.764, 33.441),
     # 'ocpn7': (-78.147, 33.911),
     '41110': (-77.715, 34.142),


     # '41109': (-77.300 ,34.484 ),
     # '41035': (-77.281 ,34.476),
     # '41036': (-76.949 , 34.207),
     # '41159': (-76.944, 34.211),
}



In [ ]:
# # Compare time series and scatter plots: Full spectrum, Partitioned spectrum, and Buoy data
# import matplotlib.pyplot as plt
# import os
# from math import radians, sin, cos, sqrt, atan2
# import sys
# sys.path.append('/home/grupos/geocean/montanoj/ShoreShop2026')
# from utils.plotting import create_text_with_metrics, fast_density_estimation

# def haversine_distance(lat1, lon1, lat2, lon2):
#     """Calculate great circle distance between two points in km"""
#     R = 6371  # Earth radius in km
#     lat1_rad = radians(lat1)
#     lat2_rad = radians(lat2)
#     dlat = radians(lat2 - lat1)
#     dlon = radians(lon2 - lon1)
#     a = sin(dlat/2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon/2)**2
#     c = 2 * atan2(sqrt(a), sqrt(1-a))
#     distance = R * c
#     return distance

# # Process each buoy
# for buoy_id, (buoy_lon, buoy_lat) in buoys.items():
#     print(f"\n{'='*60}")
#     print(f"Processing buoy: {buoy_id}")
#     print(f"Buoy coordinates: lon={buoy_lon}°, lat={buoy_lat}°")
#     print(f"{'='*60}")
    
#     try:
#         # Load buoy data
#         buoy_file = f'inputs/buoy_data/buoy_{buoy_id}_bulk_parameters.pkl'
#         if not os.path.exists(buoy_file):
#             print(f"  ⚠ Buoy file not found: {buoy_file}")
#             continue
        
#         buoy_data = pd.read_pickle(buoy_file)
#         buoy_data = buoy_data.loc['2016']  # Filter to 2016
        
#         # Find closest site in spectra
#         if 'coord_x' in spectra.coords and 'coord_y' in spectra.coords:
#             spectra_lon = spectra.coord_x.values
#             spectra_lat = spectra.coord_y.values
#         else:
#             print("  ⚠ Could not find coord_x/coord_y in spectra")
#             continue
        
#         # Convert buoy lon to 0-360 range if needed
#         if buoy_lon < 0:
#             buoy_lon_360 = buoy_lon + 360
#         else:
#             buoy_lon_360 = buoy_lon
        
#         distances_km = np.array([haversine_distance(buoy_lat, buoy_lon_360, lat, lon)
#                                   for lat, lon in zip(spectra_lat, spectra_lon)])
        
#         closest_site_idx = np.argmin(distances_km)
#         closest_distance = distances_km[closest_site_idx]
#         closest_lon = spectra_lon[closest_site_idx]
#         closest_lat = spectra_lat[closest_site_idx]
        
#         print(f"  Closest site: index={closest_site_idx}, distance={closest_distance:.3f} km")
#         print(f"  Site coordinates: lon={closest_lon:.3f}°, lat={closest_lat:.3f}°")
        
#         # Select site from spectra
#         spectra_site = spectra.isel(site=closest_site_idx)
        
#         # Compute hs, tp, dp from full spectrum
#         print("  Computing bulk parameters from full spectrum...")
#         spectra_ds = xr.Dataset({'efth': spectra_site.efth})
#         hs_full = spectra_ds.spec.hs().load()
#         tp_full = spectra_ds.spec.tp().load()
#         dp_full = spectra_ds.spec.dpm().load()
        
#         # Compute hs from partitioned spectrum (sum of all partitions)
#         print("  Computing hs from partitioned spectrum...")
#         dspart_site = dspart.isel(site=closest_site_idx)
#         hs_partitioned = dspart_site.sum("part").spec.hs().load()
        
#         # Align time indices between buoy and spectra
#         buoy_times = pd.to_datetime(buoy_data.index)
#         spectra_times = pd.to_datetime(spectra_site.time.values)
        
#         # Round buoy times to nearest hour for matching
#         buoy_times_rounded = buoy_times.round('1H')
        
#         # Find common times
#         common_times = spectra_times[spectra_times.isin(buoy_times_rounded)]
        
#         if len(common_times) == 0:
#             print(f"  ⚠ No common times found between buoy and spectra")
#             continue
        
#         print(f"  Found {len(common_times)} common time steps")
        
#         # Get indices for common times
#         spectra_time_indices = [np.where(spectra_times == t)[0][0] for t in common_times if t in spectra_times]
#         buoy_time_indices = [np.where(buoy_times_rounded == t)[0][0] for t in common_times if t in buoy_times_rounded]
        
#         # Extract aligned data
#         hs_full_aligned = hs_full.values[spectra_time_indices]
#         tp_full_aligned = tp_full.values[spectra_time_indices]
#         dp_full_aligned = dp_full.values[spectra_time_indices]
#         hs_partitioned_aligned = hs_partitioned.values[spectra_time_indices]
        
#         # Extract buoy data
#         hs_buoy = buoy_data['Hs_Buoy'].values[buoy_time_indices] if 'Hs_Buoy' in buoy_data.columns else None
#         tp_buoy = buoy_data['Tp_Buoy'].values[buoy_time_indices] if 'Tp_Buoy' in buoy_data.columns else None
#         dp_buoy = buoy_data['Dir_Buoy'].values[buoy_time_indices] if 'Dir_Buoy' in buoy_data.columns else None
        
#         # Create time series plot (3 panels: hs, tp, dir)
#         print("  Creating time series plot...")
#         fig1, axes1 = plt.subplots(3, 1, figsize=(14, 10))
#         fig1.patch.set_facecolor('black')
        
#         # Panel 1: Hs time series
#         ax1 = axes1[0]
#         ax1.set_facecolor('black')
#         if hs_buoy is not None:
#             ax1.plot(common_times, hs_buoy, color='white', label='Buoy', linewidth=1.5, alpha=0.9)
#         ax1.plot(common_times, hs_full_aligned, color='#FF69B4', label='Full Spectrum', linewidth=1.5, alpha=0.8)
#         ax1.plot(common_times, hs_partitioned_aligned, color='#4169E1', label='Partitioned', linewidth=1.5, alpha=0.8)
#         ax1.set_ylabel('Hs [m]', color='white', fontsize=12)
#         ax1.set_xlabel('Time', color='white', fontsize=12)
#         ax1.tick_params(colors='white')
#         ax1.grid(True, alpha=0.3, color='white')
#         ax1.legend(loc='upper left', facecolor='black', edgecolor='white', labelcolor='white')
#         ax1.spines['bottom'].set_color('white')
#         ax1.spines['top'].set_color('white')
#         ax1.spines['right'].set_color('white')
#         ax1.spines['left'].set_color('white')
        
#         # Panel 2: Tp time series
#         ax2 = axes1[1]
#         ax2.set_facecolor('black')
#         if tp_buoy is not None:
#             ax2.plot(common_times, tp_buoy, color='white', label='Buoy', linewidth=1.5, alpha=0.9)
#         ax2.plot(common_times, tp_full_aligned, color='#FF69B4', label='Full Spectrum', linewidth=1.5, alpha=0.8)
#         ax2.set_ylabel('Tp [s]', color='white', fontsize=12)
#         ax2.set_xlabel('Time', color='white', fontsize=12)
#         ax2.tick_params(colors='white')
#         ax2.grid(True, alpha=0.3, color='white')
#         ax2.legend(loc='upper left', facecolor='black', edgecolor='white', labelcolor='white')
#         ax2.spines['bottom'].set_color('white')
#         ax2.spines['top'].set_color('white')
#         ax2.spines['right'].set_color('white')
#         ax2.spines['left'].set_color('white')
        
#         # Panel 3: Dir time series
#         ax3 = axes1[2]
#         ax3.set_facecolor('black')
#         if dp_buoy is not None:
#             ax3.plot(common_times, dp_buoy, color='white', label='Buoy', linewidth=1.5, alpha=0.9)
#         ax3.plot(common_times, dp_full_aligned, color='#FF69B4', label='Full Spectrum', linewidth=1.5, alpha=0.8)
#         ax3.set_ylabel('Dir [°]', color='white', fontsize=12)
#         ax3.set_xlabel('Time', color='white', fontsize=12)
#         ax3.tick_params(colors='white')
#         ax3.grid(True, alpha=0.3, color='white')
#         ax3.legend(loc='upper left', facecolor='black', edgecolor='white', labelcolor='white')
#         ax3.spines['bottom'].set_color('white')
#         ax3.spines['top'].set_color('white')
#         ax3.spines['right'].set_color('white')
#         ax3.spines['left'].set_color('white')
        
#         fig1.suptitle(f'Wave Validation {buoy_id}', color='white', fontsize=14, fontweight='bold')
#         plt.tight_layout()
        
#         # Create scatter plot (4 panels)
#         print("  Creating scatter plot...")
#         fig2, axes2 = plt.subplots(2, 2, figsize=(14, 12))
#         fig2.patch.set_facecolor('black')
        
#         # Panel 1: Hs Buoy vs Hs Full Spectrum
#         ax1 = axes2[0, 0]
#         ax1.set_facecolor('black')
#         if hs_buoy is not None:
#             # Use density coloring
#             finite_mask = np.isfinite(hs_buoy) & np.isfinite(hs_full_aligned)
#             if finite_mask.sum() > 0:
#                 density = fast_density_estimation(hs_buoy[finite_mask], hs_full_aligned[finite_mask])
#                 scatter = ax1.scatter(hs_buoy[finite_mask], hs_full_aligned[finite_mask], 
#                                      c=density, cmap='plasma', s=1, alpha=0.6)
#                 cbar = plt.colorbar(scatter, ax=ax1)
#                 cbar.set_label('Density', color='white')
#                 cbar.ax.tick_params(colors='white')
                
#                 # Add metrics text
#                 metrics_text = create_text_with_metrics(hs_buoy[finite_mask], hs_full_aligned[finite_mask])
#                 ax1.text(0.05, 0.95, metrics_text, transform=ax1.transAxes,
#                         verticalalignment='top', color='white', fontsize=9,
#                         bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
        
#         max_hs = max(np.nanmax(hs_buoy) if hs_buoy is not None else 0, 
#                      np.nanmax(hs_full_aligned) if hs_full_aligned is not None else 0)
#         ax1.plot([0, max_hs], [0, max_hs], 'w--', linewidth=1.5, alpha=0.8)
#         ax1.set_xlabel('Hs - Buoy [m]', color='white', fontsize=11)
#         ax1.set_ylabel('Hs - Full Spectrum [m]', color='white', fontsize=11)
#         ax1.tick_params(colors='white')
#         ax1.grid(True, alpha=0.3, color='white')
#         ax1.spines['bottom'].set_color('white')
#         ax1.spines['top'].set_color('white')
#         ax1.spines['right'].set_color('white')
#         ax1.spines['left'].set_color('white')
        
#         # Panel 2: Hs Buoy vs Hs Partitioned
#         ax2 = axes2[0, 1]
#         ax2.set_facecolor('black')
#         if hs_buoy is not None:
#             finite_mask = np.isfinite(hs_buoy) & np.isfinite(hs_partitioned_aligned)
#             if finite_mask.sum() > 0:
#                 density = fast_density_estimation(hs_buoy[finite_mask], hs_partitioned_aligned[finite_mask])
#                 scatter = ax2.scatter(hs_buoy[finite_mask], hs_partitioned_aligned[finite_mask],
#                                      c=density, cmap='plasma', s=1, alpha=0.6)
#                 cbar = plt.colorbar(scatter, ax=ax2)
#                 cbar.set_label('Density', color='white')
#                 cbar.ax.tick_params(colors='white')
                
#                 metrics_text = create_text_with_metrics(hs_buoy[finite_mask], hs_partitioned_aligned[finite_mask])
#                 ax2.text(0.05, 0.95, metrics_text, transform=ax2.transAxes,
#                         verticalalignment='top', color='white', fontsize=9,
#                         bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
        
#         max_hs = max(np.nanmax(hs_buoy) if hs_buoy is not None else 0,
#                      np.nanmax(hs_partitioned_aligned) if hs_partitioned_aligned is not None else 0)
#         ax2.plot([0, max_hs], [0, max_hs], 'w--', linewidth=1.5, alpha=0.8)
#         ax2.set_xlabel('Hs - Buoy [m]', color='white', fontsize=11)
#         ax2.set_ylabel('Hs - Partitioned [m]', color='white', fontsize=11)
#         ax2.tick_params(colors='white')
#         ax2.grid(True, alpha=0.3, color='white')
#         ax2.spines['bottom'].set_color('white')
#         ax2.spines['top'].set_color('white')
#         ax2.spines['right'].set_color('white')
#         ax2.spines['left'].set_color('white')
        
#         # Panel 3: Tp Buoy vs Tp Full Spectrum
#         ax3 = axes2[1, 0]
#         ax3.set_facecolor('black')
#         if tp_buoy is not None:
#             finite_mask = np.isfinite(tp_buoy) & np.isfinite(tp_full_aligned)
#             if finite_mask.sum() > 0:
#                 density = fast_density_estimation(tp_buoy[finite_mask], tp_full_aligned[finite_mask])
#                 scatter = ax3.scatter(tp_buoy[finite_mask], tp_full_aligned[finite_mask],
#                                      c=density, cmap='plasma', s=1, alpha=0.6)
#                 cbar = plt.colorbar(scatter, ax=ax3)
#                 cbar.set_label('Density', color='white')
#                 cbar.ax.tick_params(colors='white')
                
#                 metrics_text = create_text_with_metrics(tp_buoy[finite_mask], tp_full_aligned[finite_mask])
#                 ax3.text(0.05, 0.95, metrics_text, transform=ax3.transAxes,
#                         verticalalignment='top', color='white', fontsize=9,
#                         bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
        
#         max_tp = max(np.nanmax(tp_buoy) if tp_buoy is not None else 0,
#                      np.nanmax(tp_full_aligned) if tp_full_aligned is not None else 0)
#         ax3.plot([0, max_tp], [0, max_tp], 'w--', linewidth=1.5, alpha=0.8)
#         ax3.set_xlabel('Tp - Buoy [s]', color='white', fontsize=11)
#         ax3.set_ylabel('Tp - Full Spectrum [s]', color='white', fontsize=11)
#         ax3.tick_params(colors='white')
#         ax3.grid(True, alpha=0.3, color='white')
#         ax3.spines['bottom'].set_color('white')
#         ax3.spines['top'].set_color('white')
#         ax3.spines['right'].set_color('white')
#         ax3.spines['left'].set_color('white')
        
#         # Panel 4: Dir Buoy vs Dir Full Spectrum
#         ax4 = axes2[1, 1]
#         ax4.set_facecolor('black')
#         if dp_buoy is not None:
#             finite_mask = np.isfinite(dp_buoy) & np.isfinite(dp_full_aligned)
#             if finite_mask.sum() > 0:
#                 density = fast_density_estimation(dp_buoy[finite_mask], dp_full_aligned[finite_mask])
#                 scatter = ax4.scatter(dp_buoy[finite_mask], dp_full_aligned[finite_mask],
#                                      c=density, cmap='plasma', s=1, alpha=0.6)
#                 cbar = plt.colorbar(scatter, ax=ax4)
#                 cbar.set_label('Density', color='white')
#                 cbar.ax.tick_params(colors='white')
                
#                 metrics_text = create_text_with_metrics(dp_buoy[finite_mask], dp_full_aligned[finite_mask])
#                 ax4.text(0.05, 0.95, metrics_text, transform=ax4.transAxes,
#                         verticalalignment='top', color='white', fontsize=9,
#                         bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
        
#         max_dir = max(np.nanmax(dp_buoy) if dp_buoy is not None else 0,
#                       np.nanmax(dp_full_aligned) if dp_full_aligned is not None else 0)
#         ax4.plot([0, max_dir], [0, max_dir], 'w--', linewidth=1.5, alpha=0.8)
#         ax4.set_xlabel('Dir - Buoy [°]', color='white', fontsize=11)
#         ax4.set_ylabel('Dir - Full Spectrum [°]', color='white', fontsize=11)
#         ax4.tick_params(colors='white')
#         ax4.grid(True, alpha=0.3, color='white')
#         ax4.spines['bottom'].set_color('white')
#         ax4.spines['top'].set_color('white')
#         ax4.spines['right'].set_color('white')
#         ax4.spines['left'].set_color('white')
        
#         fig2.suptitle(f'Wave Validation Scatter {buoy_id}', color='white', fontsize=14, fontweight='bold')
#         plt.tight_layout()
        
#         print(f"  ✓ Plots created for buoy {buoy_id}")
        
#     except Exception as e:
#         print(f"  ✗ Error processing buoy {buoy_id}: {e}")
#         import traceback
#         traceback.print_exc()
#         continue

# print("\n" + "="*60)
# print("All buoy comparisons completed!")
# print("="*60)
